# End-to-End Data Validation (PySpark)
### What this validates ✅

- ✔ Row counts
- ✔ Primary key completeness
- ✔ Record-level equality
- ✔ Transformation correctness
- ✔ Aggregation correctness
- ✔ Audit-ready mismatch output

# 1️⃣ Assumptions
- Source → raw data
- Target → transformed data
- Primary Key: id
- Transformations(example):
> - full_name = first_name + last_name
> - total_amount = qty * price

In [0]:
# Read Source & Target
from pyspark.sql.functions import sha2, concat_ws, coaleace, lit, col
source_df = spark.table("db_dlt_proj.silver.customers").cache()
target_df = spark.table("db_dlt_proj.gold.dim_customers").cache()

# Row count validation
src_count = source_df.count()
tgt_count = target_df.count()

if src_count == tgt_count:
  # Primary key validation
  src_pk = source_df.select(col("id"))
  tgt_pk = target_df.select(col("id"))
  missing_in_target = src_pk.substract(tgt_pk)
  extra_in_target = tgt_pk.substract(src_pk)

  if missing_in_target.count() == 0 and extra_in_target.count() == 0:
    print("All Priamry keys are present in Target")
  else:
    raise Exception("Primary key validation failed")

  # Column level validation
  # Re-apply Transformations on Source
  expected_df = source_df\
                .withColumn("full_name_expected", concat_ws(" ", col("first_name"), col("last_name")))\
                .withColumn("total_amount_expected", col("qty") * col("price"))
  # Align Expected & Target Columns
  expected_final = expected_df.select(
    "id",
    col("full_name_expected").alias("full_name"),
    col("total_amount_expected").alias("total_amount")
  )
  target_final = target_df.select(
    "id",
    "full_name",
    "total_amount"
  )
  # Row-Level Hash Comparison (Checking values)
  cols = ["full_name","total_amount"]
  expected_hashed = expected_final.withColumn(
    "row_hash",
    sha2(concat_ws("||", *[coaleace(col(c), lit("NULL")) for c in cols]), 256))
  target_hashed = target_final.withColumn(
    "row_hash",
    sha2(concat_ws("||", *[coaleace(col(c), lit("NULL")) for c in cols]), 256))
  # Identify Mismatched Records
  mismatch_df = expected_hashed.alias("e")\
    .join(target_hashed.alias("t"), col("e.id") == col("t.id"), "inner")\
    .filter(col("e.row_hash") != col("t.row_hash"))

  if mismatch_df.count() == 0:
    print("All records are matching")
  validation_report = {
    "source_count": src_count,
    "target_count": tgt_count,
    "missing_in_target": missing_in_target.count(),
    "extra_in_target": extra_in_target.count(),
    "mismatched_records": mismatch_df.count()
    }
  print(validation_report)
  # persist the validation Failure(Auditing)
  mismatch_df.write.mode("overwrite").save("/volume/silver/customer/validations")
else:
  raise Exception("Source and Target row counts do not match")
